# Notebook 04 — Procedimientos almacenados y cursores

Cuarto sub-bloque del Tema 06. Las funciones del Notebook 03 son **deterministas y se invocan desde queries**. Los **procedimientos** son el primo distinto: se hicieron para **efectos secundarios** y **control explícito de transacciones**. No los puedes usar en un `SELECT` — se invocan con `CALL`.

También vemos **cursores explícitos**, una construcción heredada que conviene reconocer (aparecen en código legacy) pero que en código moderno casi siempre tiene un reemplazo más limpio.

**Contenido de este notebook:**

- [Setup](#setup)
- [Procedimiento vs función — la diferencia clave](#procedimiento-vs-función--la-diferencia-clave)
- [`CREATE PROCEDURE` y `CALL`](#create-procedure-y-call)
- [Parámetros `IN`, `OUT`, `INOUT`](#parámetros-in-out-inout)
- [Control de transacciones dentro de procedimientos](#control-de-transacciones-dentro-de-procedimientos)
- [Cursores explícitos — declaración y uso](#cursores-explícitos--declaración-y-uso)
- [Cursores vs `FOR` loops sobre queries](#cursores-vs-for-loops-sobre-queries)
- [Cuándo usar procedimiento, función o SQL puro](#cuándo-usar-procedimiento-función-o-sql-puro)

## Setup

In [ ]:
import importlib.util, subprocess, sys
if importlib.util.find_spec("jupysql") is None:
    subprocess.run([sys.executable, "-m", "pip", "uninstall", "-y", "ipython-sql"], check=False)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "jupysql"], check=True)
    print("⚠ JupySQL instalado. REINICIA el kernel y vuelve a correr.")
else:
    print("✓ JupySQL listo.")

In [ ]:
%load_ext sql

from sqlalchemy import create_engine

AURORA_HOST     = "aurora-mod4.cluster-xxxxx.us-east-1.rds.amazonaws.com"
AURORA_PASSWORD = "TU_PASSWORD_AQUI"
AURORA_DATABASE = "northwind"

engine = create_engine(
    f"postgresql+psycopg2://postgres:{AURORA_PASSWORD}@{AURORA_HOST}:5432/{AURORA_DATABASE}"
)

%sql engine

## Procedimiento vs función — la diferencia clave

PostgreSQL introdujo `CREATE PROCEDURE` en la versión 11 (2018). Antes solo había funciones. Las diferencias importantes:

| Aspecto | Función (`CREATE FUNCTION`) | Procedimiento (`CREATE PROCEDURE`) |
|---|---|---|
| Cómo se invoca | `SELECT mi_funcion(args)` | `CALL mi_procedimiento(args);` |
| ¿Devuelve valor? | Sí (escalar, set, tabla) | No directamente — usa `INOUT`/`OUT` |
| ¿Usable en `SELECT`/`WHERE`/`JOIN`? | ✅ Sí | ❌ No |
| ¿Puede hacer `COMMIT`/`ROLLBACK` interno? | ❌ No | ✅ Sí |
| Caso de uso típico | Cómputo determinista, reutilizable en queries | Lógica con efectos secundarios, batch jobs |

**La clave:** si tu lógica es "calcula algo y devuélvemelo" → **función**. Si tu lógica es "haz una serie de acciones (modificar tablas, validar, registrar) y opcionalmente devuelve algunos valores" → **procedimiento**.

Casos típicos de procedimiento: refrescar agregados pre-calculados, importar un batch de datos con validaciones, ejecutar mantenimiento periódico (limpieza de tablas temporales).

## `CREATE PROCEDURE` y `CALL`

Sintaxis:

```sql
CREATE PROCEDURE nombre(parametros) AS $$
BEGIN
    -- código
END;
$$ LANGUAGE plpgsql;

-- Invocar:
CALL nombre(args);
```

Nota la ausencia de `RETURNS tipo` — un procedimiento puede devolver valores a través de parámetros `OUT`, pero no "devuelve" en el sentido funcional.

In [ ]:
%%sql
CREATE OR REPLACE PROCEDURE registrar_evento(
    p_tipo    TEXT,
    p_mensaje TEXT
) AS $$
BEGIN
    RAISE NOTICE 'Evento [%]: %', p_tipo, p_mensaje;
    -- En la vida real aquí harías INSERT INTO log_eventos ...
END;
$$ LANGUAGE plpgsql;

In [ ]:
%%sql
CALL registrar_evento('INFO', 'Procedimiento ejecutado correctamente');

## Parámetros `IN`, `OUT`, `INOUT`

Los parámetros de un procedimiento (o función) pueden tener tres modos:

- **`IN`** (default) — solo entrada. Lo más común. Pasas un valor al procedimiento.
- **`OUT`** — solo salida. El procedimiento le asigna un valor que el llamador recibe.
- **`INOUT`** — entrada y salida. Pasas un valor, el procedimiento lo modifica.

**Cómo recibir valores de un `OUT`:** desde SQL puro, los argumentos `OUT` se ven en el resultado del `CALL`.

```sql
CREATE PROCEDURE dividir(
    p_a   NUMERIC,
    p_b   NUMERIC,
    OUT p_resultado NUMERIC,
    OUT p_resto     NUMERIC
) AS $$
BEGIN
    p_resultado := p_a / p_b;
    p_resto     := p_a % p_b;
END;
$$ LANGUAGE plpgsql;

CALL dividir(17, 5, NULL, NULL);     -- los NULL son placeholders para los OUT
-- Resultado: p_resultado=3.4..., p_resto=2
```

In [ ]:
%%sql
CREATE OR REPLACE PROCEDURE dividir(
    p_a NUMERIC,
    p_b NUMERIC,
    OUT p_cociente NUMERIC,
    OUT p_resto    NUMERIC
) AS $$
BEGIN
    IF p_b = 0 THEN
        RAISE EXCEPTION 'No se puede dividir entre cero';
    END IF;
    p_cociente := p_a / p_b;
    p_resto    := p_a - (p_cociente * p_b);
END;
$$ LANGUAGE plpgsql;

In [ ]:
%%sql
CALL dividir(17, 5, NULL, NULL);

## Control de transacciones dentro de procedimientos

**Esta es la capacidad distintiva** de procedimientos vs funciones. Un procedimiento puede ejecutar `COMMIT` y `ROLLBACK` mid-ejecución. Una función no.

Caso típico: procesar un batch de N filas, commiteando cada K para no mantener una transacción gigante.

```sql
CREATE PROCEDURE procesar_batch() AS $$
DECLARE
    fila RECORD;
    n    INTEGER := 0;
BEGIN
    FOR fila IN SELECT * FROM tabla_grande LOOP
        -- ... procesar fila ...
        n := n + 1;
        IF n % 1000 = 0 THEN
            COMMIT;
            RAISE NOTICE 'Committeadas % filas', n;
        END IF;
    END LOOP;
    COMMIT;
END;
$$ LANGUAGE plpgsql;
```

**Limitaciones importantes:**

- `COMMIT`/`ROLLBACK` solo funcionan si el procedimiento se llama **fuera de una transacción explícita**. Si tu sesión ya está en una transacción (cosa común desde clientes que abren transacción automática), falla.
- Las funciones siempre corren dentro de una transacción y NO pueden hacer commit/rollback.

Para nuestro Aurora del módulo, los `CALL` simples desde JupySQL típicamente funcionan porque JupySQL no envuelve automáticamente en transacción.

In [ ]:
%%sql
-- Ejemplo conceptual — no recarga datos, solo muestra estructura
CREATE OR REPLACE PROCEDURE simular_carga_por_chunks(
    p_total       INTEGER,
    p_chunk_size  INTEGER DEFAULT 5
) AS $$
DECLARE
    n INTEGER := 0;
BEGIN
    WHILE n < p_total LOOP
        n := n + 1;
        IF n % p_chunk_size = 0 THEN
            -- En código real: COMMIT;
            RAISE NOTICE 'Checkpoint en %: aquí harías COMMIT', n;
        END IF;
    END LOOP;
    RAISE NOTICE 'Total procesado: %', n;
END;
$$ LANGUAGE plpgsql;

CALL simular_carga_por_chunks(15, 5);

## Cursores explícitos — declaración y uso

Un **cursor** es un puntero a una query que se procesa fila a fila. PL/pgSQL tiene cursores **implícitos** (los del `FOR ... IN SELECT` del Notebook 02) y **explícitos**:

```sql
DECLARE
    mi_cursor CURSOR FOR SELECT col FROM tabla WHERE ...;
BEGIN
    OPEN mi_cursor;
    LOOP
        FETCH mi_cursor INTO variable;
        EXIT WHEN NOT FOUND;
        -- usar variable
    END LOOP;
    CLOSE mi_cursor;
END;
```

Las operaciones son:

- **`OPEN`** — ejecuta la query y posiciona el cursor antes de la primera fila.
- **`FETCH ... INTO`** — avanza una fila y guarda los valores en variables.
- **`CLOSE`** — libera recursos.

**`FOUND`** se setea automáticamente: `false` cuando `FETCH` no trae más filas (final del set).

In [ ]:
%%sql
DO $$
DECLARE
    cur_categorias CURSOR FOR
        SELECT category_name FROM northwind_dwh.dim_product
        GROUP BY category_name ORDER BY 1;
    cat TEXT;
    n   INTEGER := 0;
BEGIN
    OPEN cur_categorias;
    LOOP
        FETCH cur_categorias INTO cat;
        EXIT WHEN NOT FOUND;
        n := n + 1;
        RAISE NOTICE '%. %', n, cat;
    END LOOP;
    CLOSE cur_categorias;
    
    RAISE NOTICE 'Total: %', n;
END
$$;

## Cursores vs `FOR` loops sobre queries

El mismo trabajo del ejemplo anterior se puede expresar mucho más limpio con un `FOR` loop sobre query:

```sql
DO $$
DECLARE
    n INTEGER := 0;
BEGIN
    FOR cat IN SELECT category_name FROM ... LOOP
        n := n + 1;
        RAISE NOTICE '%. %', n, cat.category_name;
    END LOOP;
    RAISE NOTICE 'Total: %', n;
END
$$;
```

Por debajo, el `FOR ... IN SELECT` crea y maneja un cursor implícito. La diferencia es que tú no tienes que escribir `OPEN`/`FETCH`/`CLOSE`.

**Cuándo conviene cursor explícito:**

- Cuando necesitas pasarlo como argumento a otra función (los cursores se pueden devolver de funciones — patrón llamado *refcursor*).
- Para procesamiento con `FETCH` parametrizado (`FETCH 10 FROM cur` lee 10 a la vez).
- Para movimientos no-secuenciales: `FETCH PRIOR`, `FETCH FIRST`, `FETCH LAST` (con cursores `SCROLL`).
- Compatibilidad con código heredado.

**Cuándo NO usar cursor (la mayoría):** prefiere `FOR ... IN SELECT`. Es más legible y maneja la limpieza automáticamente.

**Lo más importante:** si tu intención era hacer una transformación masiva (`UPDATE`, `INSERT INTO ... SELECT`), **ningún cursor es la respuesta correcta**. Una sola query SQL es 100× más rápida y mucho más simple.

## Cuándo usar procedimiento, función o SQL puro

Tabla de decisión rápida:

| Caso | Herramienta correcta |
|---|---|
| Cálculo determinista que quieres usar en muchas queries | **Función `LANGUAGE sql IMMUTABLE`** |
| Función con lógica condicional que devuelve un valor | **Función `LANGUAGE plpgsql`** |
| "Reporte" que devuelve un set de filas calculado | **Función `RETURNS TABLE`** |
| Refrescar agregados pre-calculados (DDL/DML) | **Procedimiento** |
| Cargar batch con manejo de errores y commits intermedios | **Procedimiento** |
| Transformación masiva `UPDATE ... FROM` o `INSERT ... SELECT` | **SQL puro — sin función ni procedimiento** |
| Iterar fila por fila para procesar cada una distinta | **`FOR ... IN SELECT`** dentro de procedimiento |
| Procesar fila por fila con `OPEN`/`FETCH` explícitos | **Cursor — solo si tienes razón específica** |

**Regla mnemónica:**

1. Si SQL puro lo expresa → SQL puro (rapidísimo, optimizable).
2. Si necesitas cómputo reutilizable que entra en queries → función (`IMMUTABLE`/`STABLE`).
3. Si necesitas efectos secundarios o control de transacciones → procedimiento.
4. Cursor solo cuando hay razón muy específica (procesamiento en lotes con `FETCH N`, scrollable, compatibilidad).

In [ ]:
%%sql
-- Limpieza de objetos creados en este notebook
DROP PROCEDURE IF EXISTS registrar_evento(TEXT, TEXT);
DROP PROCEDURE IF EXISTS dividir(NUMERIC, NUMERIC);
DROP PROCEDURE IF EXISTS simular_carga_por_chunks(INTEGER, INTEGER);

## Cierre

Lo que cubriste:

| Tema | Construcción clave |
|---|---|
| Crear procedimiento | `CREATE PROCEDURE nombre(params) AS $$ ... $$ LANGUAGE plpgsql;` |
| Invocar | `CALL nombre(args);` — no `SELECT` |
| Parámetros con dirección | `IN` (default), `OUT`, `INOUT` |
| Control de transacciones | `COMMIT`/`ROLLBACK` dentro del procedimiento (con restricciones) |
| Cursor explícito | `DECLARE c CURSOR FOR ...; OPEN c; FETCH c INTO v; CLOSE c;` |
| Cursor implícito | `FOR row IN SELECT ... LOOP ... END LOOP;` — la opción limpia |

El siguiente notebook (**05 — Práctica**) consolida lo de los cuatro notebooks anteriores con ejercicios graduales sobre Northwind DWH.

---

<p align="center">
<a href="03_funciones.ipynb">← Anterior: Notebook 03</a> | <a href="Readme.md">Volver al índice</a> | <a href="05_practica.ipynb">Siguiente: Notebook 05 — Práctica →</a>
</p>